# Chuẩn đoán bệnh tiểu đường

Trong phần trước, chúng ta đã tiến hành các bước sau:
1. Định nghĩa vấn đề
    + Mô tả vấn đề
    + Xác định đầu vào, đầu ra, loại bài toán
2. Chuẩn bị vấn đề:
    + Tải thư viện và dữ liệu
3. Phân tích dữ liệu:
    + Hiển thị thông tin dữ liệu
    + Thông tin thống kê trên các thuộc tính dữ liệu
    + Mối tương quan giữa các thuộc tính
5. Chia dữ liệu
6. Chuẩn bị dữ liệu:
    + Làm sạch dữ liệu (tạo bảng dữ liệu chỉ có thuộc tính nhập, xuất, xử lý dữ liệu thiếu và trùng lặp)
    + Biến đổi dữ liệu (chuẩn hóa)


**Kết quả:**
+ exps/data:
  + train.xlsx
  + test.xls
+ feature1:
  
  + scale_columns.npz (chứa cột biến đổi)

  + Feature MinMax:
    + minmax_scaler.joblib
    + feat_minmax.npz
    + df_minmax.xlsx

+ Feature Standard:
    + standard_scaler.joblib
    + feat_standard.npz
    + df_standard.xlsx

## Khởi tạo thí nghiệm 

### Khai báo thư viện

In [2]:
import os, sys
from IPython import display
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import ticker

import pandas as pd
import seaborn as sns
import joblib
import pprint
import random

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV

from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
import sklearn

from sklearn.metrics import accuracy_score , ConfusionMatrixDisplay, confusion_matrix

import warnings
%matplotlib inline


warnings.filterwarnings("ignore")

c:\Python313\Lib\site-packages\xgboost\compat.py:105: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### Tham số thực nghiệm

In [3]:
params = {}

params['exps_dir'] = '../exps'
params['exp_name'] = 'pima_standard'

params['exp_root'] = f'{params["exps_dir"]}/result1_standard'
params["save_dir"]  = f'{params["exps_dir"]}/result1_{params["exp_name"]}'

params["data_path"]  = f'{params["exps_dir"]}/feature1/df_standard.xlsx'

params['k_fold'] = 5
params['random_state'] = 42

print("params: ")
for k in params: print(f'+ {k}: {params[k]}')

random.seed(params['random_state'])
os.environ['PYTHONHASHSEED'] = str(params['random_state'])
np.random.seed(params['random_state'])

params: 
+ exps_dir: ../exps
+ exp_name: pima_standard
+ exp_root: ../exps/result1_standard
+ save_dir: ../exps/result1_pima_standard
+ data_path: ../exps/feature1/df_standard.xlsx
+ k_fold: 5
+ random_state: 42


## 5. Dữ liệu kiểm nghiệm
Chuẩn bị dữ liệu kiểm nghiệm theo phương pháp hold-out:
- Chia tập dữ liệu thành 2 phần train/test với tỉ lệ 7/3
- Tập train sẽ được dùng để huấn luyện mô hình và điều chỉnh tham số, với hai cách: 
    - Hold-out (tiếp tục chia 7/3 với train/valid)
    - k-fold (chia thành k phần đều nhau với k-1 phần cho train/1 phần cho valid)
    - Trong đó, train là dùng huấn luyện và valid để tối ưu tham số
- Tập test dùng để kiểm nghiệm lại độ hiệu quả của thuật toán sau khi chọn mô hình tối ưu

In [4]:
# tải dữ liệu
df = pd.read_excel(params['data_path'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 537 entries, 0 to 536
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               537 non-null    float64
 1   Glucose                   537 non-null    float64
 2   BloodPressure             537 non-null    float64
 3   SkinThickness             537 non-null    float64
 4   Insulin                   537 non-null    float64
 5   BMI                       537 non-null    float64
 6   DiabetesPedigreeFunction  537 non-null    float64
 7   Age                       537 non-null    float64
 8   Outcome                   537 non-null    int64  
dtypes: float64(8), int64(1)
memory usage: 37.9 KB


In [5]:
# Chia dữ liệu thành input/ouptu
x_train, y_train = df.values[:,:-1], df.values[:,-1].astype(int)
print(f'x_shape: {x_train.shape}, y_shape: {y_train.shape})')
print('Input: \n', x_train[:10,:])
print('Output: \n', y_train[:10])

x_shape: (537, 8), y_shape: (537,))
Input: 
 [[-8.36294303e-01 -8.96200501e-01 -1.00392807e+00 -1.26954457e+00
  -9.56993000e-01 -1.20379368e+00 -6.14216360e-01 -9.48610283e-01]
 [ 3.90727666e-01 -5.64089421e-01 -1.97904130e-02  2.96930903e-02
   2.13679743e+00  6.64529988e-01 -9.09737865e-01 -4.34666726e-01]
 [-1.14304979e+00  4.32243819e-01 -3.47836300e-01  1.56515578e+00
   1.26775517e+00  1.44060290e+00 -3.06991033e-01 -7.77295764e-01]
 [ 8.39721738e-02  2.99399387e-01 -3.47836300e-01 -9.15207028e-01
   2.94427848e-01  1.18404607e-01 -9.06811910e-01 -4.34666726e-01]
 [-8.36294303e-01 -6.30511637e-01 -3.46427222e+00  1.09270572e+00
  -6.67312248e-01  1.58432010e+00 -8.39514933e-01 -6.38042901e-03]
 [-5.29538810e-01 -1.32794491e+00 -1.66001985e+00 -7.97094513e-01
  -2.96520886e-01 -5.42694538e-01  3.59623360e+00 -6.91638505e-01]
 [-2.22783318e-01  1.99766063e-01  4.72278417e-01  2.96930903e-02
  -1.80648585e-01 -1.60620186e+00 -5.90808716e-01  1.87807928e+00]
 [-8.36294303e-01  4.994

## 6. Lượng giá thuật toán 

### 6.1 Baselines

In [6]:
# kfold = KFold(n_splits=params['k_fold'], shuffle=True, random_state=params['random_state'])
# for fold, (train_idx, valid_idx) in enumerate(kfold.split(x_train, y_train)):
#     print(f'Fold {fold}: ')
#     print(f'+ train_idx: {train_idx}')
#     print(f'+ valid_idx: {valid_idx}')
#     print(f'+ train / valid: {valid_idx}')
#     pass

In [7]:
baseline_models = {
    'Logistic Regression': LogisticRegression(random_state=params["random_state"]),
    # 'Decision Tree': DecisionTreeClassifier(random_state=params["random_state"]),
    'Linear Discriminant Analysis': LinearDiscriminantAnalysis(),
    # 'K-Nearest Neighbors': KNeighborsClassifier(),
    'SVM': SVC(random_state=params["random_state"]),
    'XGBoost': XGBClassifier(random_state=params["random_state"]),
    'Random Forest': RandomForestClassifier(random_state=params["random_state"]),
}


kfold = KFold(n_splits=params['k_fold'], shuffle=True, random_state=params['random_state'])
for name, model in baseline_models.items():
    print (name)
    acc = cross_val_score(model, x_train, y_train, cv=kfold, scoring='accuracy').mean()
    f1 = cross_val_score(model, x_train, y_train, cv=kfold, scoring='f1').mean()
    auc = cross_val_score(model, x_train, y_train, cv=kfold, scoring='roc_auc').mean()
    print(f'+ Accuracy: {acc:.4f}\n F1-score: {f1:.4f}\n AUC: {auc:.4f}')
    print('-'*20)



Logistic Regression
+ Accuracy: 0.7859
 F1-score: 0.6562
 AUC: 0.8530
--------------------
Linear Discriminant Analysis
+ Accuracy: 0.7822
 F1-score: 0.6493
 AUC: 0.8535
--------------------
SVM
+ Accuracy: 0.7616
 F1-score: 0.5964
 AUC: 0.8468
--------------------
XGBoost
+ Accuracy: 0.7542
 F1-score: 0.6334
 AUC: nan
--------------------
Random Forest
+ Accuracy: 0.7691
 F1-score: 0.6482
 AUC: 0.8425
--------------------


### 6.2 Tinh chỉnh mô hình

In [8]:
tunning_results = {
    "best_clf"   : {},
    "best_score" : {},
}

tunning_models  = {}
tunning_params  = {}

# khởi tạo các tham số mặc định
tunning_models['Logistic Regression'] = LogisticRegression(random_state=params["random_state"])
tunning_params['Logistic Regression'] = {
    'C': [1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l1', 'l2']
}

tunning_models['Linear Discriminant Analysis'] = LinearDiscriminantAnalysis()
tunning_params['Linear Discriminant Analysis'] = {
    'solver': ['svd', 'lsqr', 'eigen'],
    'shrinkage': [None, 'auto']
}

tunning_models['Random Forest'] = RandomForestClassifier(random_state=params["random_state"])
tunning_params['Random Forest'] = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

for name, model in tunning_models.items():
    print(name)
    grid_clf = GridSearchCV(model, tunning_params[name], cv=kfold, scoring='accuracy')
    grid_result = grid_clf.fit(x_train, y_train)

    # store best model
    tunning_results["best_clf"][name] = grid_clf.best_estimator_

    # get search results
    tunning_results["best_score"][name] = grid_result.best_score_


    # information
    print(f'+ Best score: {grid_result.best_score_}')
    print(f'+ Best turnning params: {grid_result.best_params_}')
    print(f'+ Best full params: {grid_clf.best_estimator_.get_params()}')
    print()

Logistic Regression
+ Best score: 0.7858947732779509
+ Best turnning params: {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}
+ Best full params: {'C': 1, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 100, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}

Linear Discriminant Analysis
+ Best score: 0.7858601592246452
+ Best turnning params: {'shrinkage': 'auto', 'solver': 'lsqr'}
+ Best full params: {'covariance_estimator': None, 'n_components': None, 'priors': None, 'shrinkage': 'auto', 'solver': 'lsqr', 'store_covariance': False, 'tol': 0.0001}

Random Forest
+ Best score: 0.7765662859120803
+ Best turnning params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
+ Best full params: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'max_

## 7. Kiểm nghiệm kết quả trên test

In [9]:
df_test = pd.read_csv(f'{params["exps_dir"]}/data/test.csv')
df_test

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,98,58,33,190,34.0,0.430,43,0
1,2,112,75,32,0,35.7,0.148,21,0
2,2,108,64,0,0,30.8,0.158,21,0
3,8,107,80,0,0,24.6,0.856,34,0
4,7,136,90,0,0,29.9,0.210,50,0
...,...,...,...,...,...,...,...,...,...
226,0,119,0,0,0,32.4,0.141,24,1
227,4,109,64,44,99,34.8,0.905,26,1
228,0,127,80,37,210,36.3,0.804,23,0
229,6,105,70,32,68,30.8,0.122,37,0


In [10]:
standard_scaler = joblib.load(f'{params["exps_dir"]}/feature1/standard_scaler.joblib')
display.display(standard_scaler.__dict__)
scale_columns = dict(np.load(f'{params["exps_dir"]}/feature1/scale_columns.npz'))['columns']
scale_columns = scale_columns.tolist()
scale_columns

{'with_mean': True,
 'with_std': True,
 'copy': True,
 'feature_names_in_': array(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
        'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'], dtype=object),
 'n_features_in_': 8,
 'n_samples_seen_': np.int64(537),
 'mean_': array([  3.72625698, 121.98496241,  72.24131274,  28.74860335,
        140.59031657,  32.27612782,   0.46991993,  33.0744879 ]),
 'var_': array([1.06271132e+01, 9.06636647e+02, 1.48679396e+02, 7.16816787e+01,
        7.44801838e+03, 4.84152827e+01, 1.16805843e-01, 1.36292403e+02]),
 'scale_': array([ 3.25992533, 30.11040763, 12.19341611,  8.46650333, 86.30190252,
         6.95810913,  0.3417687 , 11.67443374])}

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age']

In [11]:
df_test[scale_columns] = standard_scaler.transform(df_test[scale_columns])
df_test.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,0.697483,-0.796567,-1.167951,0.502143,0.572521,0.247750,-0.116804,0.850192,0
1,-0.529539,-0.331612,0.226244,0.384031,-1.629052,0.492069,-0.941923,-1.034268,0
2,-0.529539,-0.464456,-0.675882,-3.395570,-1.629052,-0.212145,-0.912664,-1.034268,0
3,1.310994,-0.497667,0.636301,-3.395570,-1.629052,-1.103192,1.129653,0.079277,0
4,1.004239,0.465455,1.456416,-3.395570,-1.629052,-0.341490,-0.760514,1.449793,0


In [12]:
x_test, y_test = df_test.values[:,:-1], df_test.values[:,-1].astype(int)

In [13]:
# baseline models
for name, model in baseline_models.items():
    model.fit(x_train, y_train)
    y_pred_test = model.predict(x_test)
    test_acc = accuracy_score(y_test, y_pred_test)

    print(name)
    print(f'+ Test accuracy: {test_acc:.4f}')

Logistic Regression
+ Test accuracy: 0.7403
Linear Discriminant Analysis
+ Test accuracy: 0.7403
SVM
+ Test accuracy: 0.7229
XGBoost
+ Test accuracy: 0.7273
Random Forest
+ Test accuracy: 0.7100


In [14]:
# Kiểm tra lại kết quả trên tập test (tunning models)
for name, model in tunning_results["best_clf"].items():
    model.fit(x_train, y_train)
    y_pred_test = model.predict(x_test)
    test_acc = accuracy_score(y_test, y_pred_test)

    print(name)
    print(f'+ Test accuracy: {test_acc:.4f}')

Logistic Regression
+ Test accuracy: 0.7403
Linear Discriminant Analysis
+ Test accuracy: 0.7532
Random Forest
+ Test accuracy: 0.7489


## 8. Lưu kết quả thí nghiệm

In [17]:
import os
save_dir = params["save_dir"]
os.makedirs(save_dir, exist_ok=True)

# Lưu notebook thành HTML (không sử dụng $cur_dir)
!jupyter nbconvert "model1.ipynb" --to html --output-dir "{save_dir}" --output "model1"

[NbConvertApp] Converting notebook model1.ipynb to html
[NbConvertApp] Writing 336059 bytes to ..\exps\result1_pima_standard\model1.html
